In [ ]:
import pycolmap
import numpy as np
import torch
from PIL import Image

sfm_path = "../data/NeRF/nerf_synthetic/chair"
reconstruction = pycolmap.Reconstruction(f"{sfm_path}/sparse/0")


def getCameraInfo(id):
    camera = reconstruction.cameras[id]
    camera_width = camera.width
    camera_height = camera.height
    [camera_fx, camera_fy, camera_cx, camera_cy] = camera.params
    # 相机内参
    return camera_width, camera_height, camera_fx, camera_fy, camera_cx, camera_cy


def readImageData(image_name):
    image_path = f"{sfm_path}/images/{image_name}"
    data = np.array(Image.open(image_path))
    return data


def getImages():
    view_mtx_lst = []
    camera_id_lst = []
    image_data_lst = []
    the_image = None
    for image_id, image in reconstruction.images.items():
        # print(image_id, image.name)
        cam_from_world = image.cam_from_world()
        R = cam_from_world.rotation.matrix()  # 旋转矩阵
        t = cam_from_world.translation  # 平移向量
        W = np.eye(4)
        W[0:3, 0:3] = R
        W[0:3, 3] = t
        image_data = readImageData(image.name)
        image_data = image_data / 255.0
        view_mtx_lst.append(W)  # 视图矩阵
        camera_id_lst.append(image.camera_id)
        image_data_lst.append(image_data)
    return camera_id_lst, view_mtx_lst, image_data_lst

def getPointCloud():
    point3d_pos = np.array([])
    npoint3d = len(reconstruction.points3D)
    point3d_pos = np.zeros((npoint3d, 3))  # 稀疏点云的位置，世界坐标系下
    point3d_rgb = np.zeros((npoint3d, 3))  # 稀疏点云的颜色0-1
    for i, (point3D_id, point3D) in enumerate(reconstruction.points3D.items()):
        point3d_pos[i, :] = point3D.xyz
        point3d_rgb[i, :] = point3D.color / 255
    return point3d_pos, point3d_rgb

In [9]:
camera_id_lst, view_mtx_lst, image_data_lst = getImages()
image_data_lst

[array([[[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],
 
        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],
 
        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],
 
        ...,
 
        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],
 
        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]],
 
        [[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.],
         ...,
         [1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]], shape=(800, 800, 3)),
 array([[[1., 1., 1.],
         [1